In [10]:
"""
================================================================================
Feature Descriptors -> Silicide Phase Fraction
Two-layer Analysis Strategy:
  Layer 1: Pearson + Spearman correlation (all 6 features)
  Layer 2: Simple linear regression with 95% confidence band
           (only for statistically significant features, p < 0.05)
Purpose : Provide mechanistic linkage between ML descriptors and
          microstructural parameter (Silicide phase fraction)
          — Response to Reviewer Comment
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.stats import pearsonr, spearmanr, t as t_dist
import warnings
import os
warnings.filterwarnings('ignore')

# ============================================================
# Plot settings
# ============================================================
plt.rcParams['font.family']        = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi']         = 300
sns.set_style("whitegrid")

print("=" * 80)
print("Feature Descriptors -> Silicide Phase Fraction")
print("Layer 1: Pearson & Spearman Correlation")
print("Layer 2: Simple Linear Regression (significant features only, p<0.05)")
print("=" * 80)

# ============================================================
# 1. Paths & Loading
# ============================================================
DATA_FILE  = r'D:\博二上\模型分析可视化\回归分析\Kq-硅化物相比例-副本.xlsx'
OUTPUT_DIR = r'D:\博二上\模型分析可视化\回归分析\硅化物相比例-副本'
ORIGIN_DIR = os.path.join(OUTPUT_DIR, 'Origin_data')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ORIGIN_DIR, exist_ok=True)

df = pd.read_excel(DATA_FILE)
print(f"\n  Loaded: {len(df)} samples, columns: {df.columns.tolist()}")

# ============================================================
# 2. Setup
# ============================================================
FEATURE_NAMES = [
    'x',
    'ΔSmix',
    'Ω',
    'Ec',
    'deltaP_E13 Electron affinity',
    'deltaP_热导率W/(mk)'
]

# Display names for plots (plain English, no encoding risk)
FEATURE_DISPLAY = {
    'x':                             'x (Electronegativity)',
    'ΔSmix':                         'ΔSmix (Mixing Entropy)',
    'Ω':                             'Ω (Omega Parameter)',
    'Ec':                            'Ec (Cohesive Energy)',
    'deltaP_E13 Electron affinity':  'δP_EA (Electron Affinity Disparity)',
    'deltaP_热导率W/(mk)':            'δP_TC (Thermal Conductivity Disparity)',
}

TARGET       = '硅化物相体积分数'
TARGET_LABEL = 'Silicide Phase Fraction (%)'
ALLOY_COL    = 'alloy'
SIG_THRESH   = 0.05          # significance threshold for Layer 2

X         = df[FEATURE_NAMES].values
y         = df[TARGET].values
alloy_ids = df[ALLOY_COL].values
n         = len(y)

# ============================================================
# Helper functions
# ============================================================
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def sig_color(p):
    if p < 0.001: return '#D62728'   # red
    if p < 0.01:  return '#FF7F0E'   # orange
    if p < 0.05:  return '#2CA02C'   # green
    return '#AAAAAA'                  # gray

def pearson_ci(r, n, alpha=0.05):
    """Fisher z-transform 95% CI for Pearson r."""
    z   = np.arctanh(r)
    se  = 1.0 / np.sqrt(n - 3)
    zc  = stats.norm.ppf(1 - alpha / 2)
    return np.tanh(z - zc * se), np.tanh(z + zc * se)

def regression_ci_band(x_obs, y_obs, x_line, alpha=0.05):
    """
    Return (y_fit, ci_low, ci_high, pi_low, pi_high) for a given x_line.
    ci = confidence interval for the mean response
    pi = prediction interval for individual observation
    """
    n     = len(x_obs)
    x_m   = x_obs.mean()
    Sxx   = np.sum((x_obs - x_m) ** 2)
    slope, intercept, r, p, se_slope = stats.linregress(x_obs, y_obs)
    y_hat = slope * x_obs + intercept
    SSE   = np.sum((y_obs - y_hat) ** 2)
    s2    = SSE / (n - 2)           # MSE
    s     = np.sqrt(s2)

    y_line = slope * x_line + intercept

    # standard error of mean prediction at each x_line point
    se_mean = s * np.sqrt(1/n + (x_line - x_m)**2 / Sxx)
    # standard error of individual prediction
    se_ind  = s * np.sqrt(1 + 1/n + (x_line - x_m)**2 / Sxx)

    tc = t_dist.ppf(1 - alpha/2, df=n-2)

    ci_low  = y_line - tc * se_mean
    ci_high = y_line + tc * se_mean
    pi_low  = y_line - tc * se_ind
    pi_high = y_line + tc * se_ind

    return y_line, ci_low, ci_high, pi_low, pi_high, slope, intercept, r, p, s

# ============================================================
# LAYER 1: Pearson & Spearman Correlation
# ============================================================
print("\n" + "=" * 80)
print("LAYER 1: Pearson & Spearman Correlation")
print("=" * 80)

corr_rows = []
for i, feat in enumerate(FEATURE_NAMES):
    pr, pp = pearsonr(X[:, i],  y)
    sr, sp = spearmanr(X[:, i], y)
    cil, cih = pearson_ci(pr, n)
    corr_rows.append({
        'Feature':       feat,
        'Display':       FEATURE_DISPLAY[feat],
        'Pearson_r':     pr,
        'Pearson_r2':    pr ** 2,
        'Pearson_p':     pp,
        'Pearson_sig':   sig_label(pp),
        'CI95_low':      cil,
        'CI95_high':     cih,
        'Spearman_rho':  sr,
        'Spearman_p':    sp,
        'Spearman_sig':  sig_label(sp),
        'Significant':   pp < SIG_THRESH,
    })

corr_df = (pd.DataFrame(corr_rows)
           .sort_values('Pearson_r', key=abs, ascending=False)
           .reset_index(drop=True))

print("\n  Results (sorted by |Pearson r|):")
print(f"  {'Feature':<35} {'Pearson r':>10} {'p':>8} {'sig':>4}  "
      f"{'95% CI':>18}  {'Spearman ρ':>10} {'p':>8} {'sig':>4}")
print("  " + "-" * 100)
for _, row in corr_df.iterrows():
    ci_str = f"[{row['CI95_low']:+.3f}, {row['CI95_high']:+.3f}]"
    print(f"  {row['Feature']:<35} {row['Pearson_r']:>+10.4f} {row['Pearson_p']:>8.4f} "
          f"{row['Pearson_sig']:>4}  {ci_str:>18}  "
          f"{row['Spearman_rho']:>+10.4f} {row['Spearman_p']:>8.4f} {row['Spearman_sig']:>4}")

sig_features = corr_df[corr_df['Significant']]['Feature'].tolist()
print(f"\n  Significant features (p < {SIG_THRESH}): {sig_features}")

# ============================================================
# LAYER 2: Simple Linear Regression (significant features only)
# ============================================================
print("\n" + "=" * 80)
print(f"LAYER 2: Simple Linear Regression (p < {SIG_THRESH} features only)")
print("=" * 80)

reg_results = []
for feat in sig_features:
    i      = FEATURE_NAMES.index(feat)
    x_obs  = X[:, i]
    x_line = np.linspace(x_obs.min(), x_obs.max(), 300)
    y_line, ci_low, ci_high, pi_low, pi_high, slope, intercept, r, p, s = \
        regression_ci_band(x_obs, y, x_line)

    r2 = r ** 2
    n_pts = len(x_obs)
    # t-test for slope
    Sxx     = np.sum((x_obs - x_obs.mean()) ** 2)
    se_b    = s / np.sqrt(Sxx)
    t_slope = slope / se_b
    p_slope = 2 * (1 - t_dist.cdf(abs(t_slope), df=n_pts - 2))

    reg_results.append({
        'Feature':    feat,
        'Display':    FEATURE_DISPLAY[feat],
        'Slope':      slope,
        'Intercept':  intercept,
        'R':          r,
        'R2':         r2,
        'p_slope':    p_slope,
        'Sig_slope':  sig_label(p_slope),
        'SE_resid':   s,
        'x_obs':      x_obs,
        'x_line':     x_line,
        'y_line':     y_line,
        'ci_low':     ci_low,
        'ci_high':    ci_high,
        'pi_low':     pi_low,
        'pi_high':    pi_high,
    })

    print(f"\n  Feature: {feat}")
    sign = '+' if intercept >= 0 else ''
    print(f"    Equation : Silicide% = {slope:.4f} × {feat} {sign}{intercept:.4f}")
    print(f"    R        : {r:.4f}")
    print(f"    R²       : {r2:.4f}")
    print(f"    Slope t  : {t_slope:.4f}, p = {p_slope:.6f}  {sig_label(p_slope)}")
    print(f"    Residual SE: {s:.4f} %")

# ============================================================
# FIGURES
# ============================================================
print("\n" + "=" * 80)
print("Generating figures ...")
print("=" * 80)

# ------------------------------------------------------------------
# Fig 1: Correlation overview bar chart (Pearson r, all features)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
corr_plot = corr_df.sort_values('Pearson_r', key=abs, ascending=True).copy()
yp     = np.arange(len(corr_plot))
colors = [sig_color(p) for p in corr_plot['Pearson_p']]

bars = ax.barh(yp, corr_plot['Pearson_r'],
               color=colors, alpha=0.82, edgecolor='black', linewidth=0.8)

# 95% CI error bars
for i, row in enumerate(corr_plot.itertuples(index=False)):
    pr  = row.Pearson_r
    cil = row.CI95_low
    cih = row.CI95_high
    ax.errorbar(pr, i,
                xerr=[[pr - cil], [cih - pr]],
                fmt='none', color='black', capsize=4, linewidth=1.2, zorder=5)
    # significance label
    offset = 0.015
    ha = 'left' if pr >= 0 else 'right'
    ax.text(cih + offset if pr >= 0 else cil - offset, i,
            f'{row.Pearson_sig}', va='center', ha=ha, fontsize=10,
            fontweight='bold',
            color=sig_color(row.Pearson_p))

ax.set_yticks(yp)
ax.set_yticklabels([FEATURE_DISPLAY[f] for f in corr_plot['Feature']], fontsize=10)
ax.set_xlabel('Pearson Correlation Coefficient (r)', fontsize=12)
ax.set_title('Pearson Correlation: Feature Descriptors vs Silicide Phase Fraction\n'
             '(error bars = 95% CI)',
             fontsize=12, fontweight='bold')
ax.axvline(0, color='black', lw=1)
ax.set_xlim(-0.75, 0.85)
ax.grid(True, alpha=0.3, axis='x')

legend_elems = [
    mpatches.Patch(fc='#D62728', alpha=0.82, label='p < 0.001 (***)'),
    mpatches.Patch(fc='#FF7F0E', alpha=0.82, label='p < 0.01  (**)'),
    mpatches.Patch(fc='#2CA02C', alpha=0.82, label='p < 0.05  (*)'),
    mpatches.Patch(fc='#AAAAAA', alpha=0.82, label='p ≥ 0.05  (ns)'),
]
ax.legend(handles=legend_elems, loc='lower right', fontsize=9, framealpha=0.9)
plt.tight_layout()
p1 = os.path.join(OUTPUT_DIR, 'Fig1_Pearson_Correlation_Overview.png')
plt.savefig(p1, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {os.path.basename(p1)}")

# ------------------------------------------------------------------
# Fig 2: Pearson vs Spearman side-by-side (all features)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))
corr_plot2 = corr_df.sort_values('Pearson_r', key=abs, ascending=True).copy()
yp2   = np.arange(len(corr_plot2))
width = 0.35

ax.barh(yp2 - width/2, corr_plot2['Pearson_r'], width,
        color='steelblue', alpha=0.82, edgecolor='black',
        linewidth=0.7, label='Pearson r')
ax.barh(yp2 + width/2, corr_plot2['Spearman_rho'], width,
        color='darkorange', alpha=0.82, edgecolor='black',
        linewidth=0.7, label='Spearman ρ')

# Pearson CI
for i, row in enumerate(corr_plot2.itertuples(index=False)):
    pr = row.Pearson_r
    ax.errorbar(pr, i - width/2,
                xerr=[[pr - row.CI95_low], [row.CI95_high - pr]],
                fmt='none', color='black', capsize=3, linewidth=1)

# sig text
for i, row in enumerate(corr_plot2.itertuples(index=False)):
    max_r = max(abs(row.Pearson_r), abs(row.Spearman_rho)) + 0.03
    ax.text(max_r, i,
            f'{row.Pearson_sig} / {row.Spearman_sig}',
            va='center', fontsize=8.5, color='#333333')

ax.set_yticks(yp2)
ax.set_yticklabels([FEATURE_DISPLAY[f] for f in corr_plot2['Feature']], fontsize=10)
ax.set_xlabel('Correlation Coefficient', fontsize=12)
ax.set_title('Pearson vs Spearman Correlation with Silicide Phase Fraction\n'
             '(Pearson error bars = 95% CI  |  significance: Pearson / Spearman)',
             fontsize=12, fontweight='bold')
ax.axvline(0, color='black', lw=1)
ax.grid(True, alpha=0.3, axis='x')
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
p2 = os.path.join(OUTPUT_DIR, 'Fig2_Pearson_vs_Spearman.png')
plt.savefig(p2, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {os.path.basename(p2)}")

# ------------------------------------------------------------------
# Fig 3: Individual scatter + regression line + CI band + PI band
#        One subplot per significant feature
# ------------------------------------------------------------------
n_sig = len(reg_results)
ncols = min(n_sig, 2)
nrows = int(np.ceil(n_sig / ncols))
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(7 * ncols, 6 * nrows),
                         squeeze=False)

# color cycle per feature
feat_colors = ['#1F77B4', '#2CA02C', '#D62728', '#9467BD']

for idx, res in enumerate(reg_results):
    ax   = axes[idx // ncols][idx % ncols]
    fc   = feat_colors[idx % len(feat_colors)]
    feat = res['Feature']
    disp = res['Display']
    i    = FEATURE_NAMES.index(feat)

    # scatter
    ax.scatter(res['x_obs'], y,
               s=55, alpha=0.70, color=fc,
               edgecolors='white', linewidths=0.5, zorder=4,
               label='Observed data')

    # regression line
    ax.plot(res['x_line'], res['y_line'],
            color='black', lw=2.0, zorder=5,
            label=f'Fit: y={res["Slope"]:.3f}x+{res["Intercept"]:+.2f}')

    # 95% CI band
    ax.fill_between(res['x_line'], res['ci_low'], res['ci_high'],
                    alpha=0.25, color=fc, label='95% Confidence Band')

    # 95% PI band
    ax.fill_between(res['x_line'], res['pi_low'], res['pi_high'],
                    alpha=0.10, color=fc, linestyle='--',
                    label='95% Prediction Band')

    # stats text box
    pr_val, pp_val = pearsonr(res['x_obs'], y)
    sr_val, sp_val = spearmanr(res['x_obs'], y)
    textstr = (f'Pearson r = {res["R"]:+.4f} {sig_label(pp_val)}\n'
               f'Spearman ρ = {sr_val:+.4f} {sig_label(sp_val)}\n'
               f'R² = {res["R2"]:.4f}\n'
               f'p(slope) = {res["p_slope"]:.4e} {res["Sig_slope"]}')
    ax.text(0.04, 0.97, textstr,
            transform=ax.transAxes, fontsize=9.5, va='top',
            bbox=dict(boxstyle='round,pad=0.4', fc='white',
                      ec='#CCCCCC', alpha=0.90))

    ax.set_xlabel(disp, fontsize=11)
    ax.set_ylabel(TARGET_LABEL, fontsize=11)
    ax.set_title(f'{disp}\nvs Silicide Phase Fraction',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, loc='lower right', framealpha=0.85)
    ax.grid(True, alpha=0.25)

# hide empty subplots
for idx in range(n_sig, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

plt.suptitle('Simple Linear Regression: Significant Feature Descriptors vs Silicide Phase Fraction\n'
             '(Shaded bands: 95% Confidence Interval and Prediction Interval)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
p3 = os.path.join(OUTPUT_DIR, 'Fig3_SLR_Significant_Features_CI_PI.png')
plt.savefig(p3, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {os.path.basename(p3)}")

# ------------------------------------------------------------------
# Fig 4: Combined panel (publication-style)
#   Left column:  Pearson correlation overview
#   Right columns: one scatter per significant feature
# ------------------------------------------------------------------
total_panels = 1 + n_sig
fig = plt.figure(figsize=(7 * total_panels, 6))
gs  = fig.add_gridspec(1, total_panels, wspace=0.35)

# Panel (a): correlation bar
ax_corr = fig.add_subplot(gs[0, 0])
corr_p4 = corr_df.sort_values('Pearson_r', key=abs, ascending=True).copy()
yp4     = np.arange(len(corr_p4))
cl4     = [sig_color(p) for p in corr_p4['Pearson_p']]
ax_corr.barh(yp4, corr_p4['Pearson_r'],
             color=cl4, alpha=0.82, edgecolor='black', linewidth=0.8)
for i, row in enumerate(corr_p4.itertuples(index=False)):
    pr = row.Pearson_r
    ax_corr.errorbar(pr, i,
                     xerr=[[pr - row.CI95_low], [row.CI95_high - pr]],
                     fmt='none', color='black', capsize=3, linewidth=1.1)
    ax_corr.text(row.CI95_high + 0.01 if pr >= 0 else row.CI95_low - 0.01,
                 i, row.Pearson_sig,
                 va='center', ha='left' if pr >= 0 else 'right',
                 fontsize=9, fontweight='bold',
                 color=sig_color(row.Pearson_p))
ax_corr.set_yticks(yp4)
ax_corr.set_yticklabels([FEATURE_DISPLAY[f] for f in corr_p4['Feature']], fontsize=9)
ax_corr.set_xlabel('Pearson r', fontsize=11)
ax_corr.set_title('(a) Pearson Correlation\n(error bars = 95% CI)',
                  fontsize=11, fontweight='bold')
ax_corr.axvline(0, color='black', lw=0.8)
ax_corr.grid(True, alpha=0.3, axis='x')
legend_elems2 = [
    mpatches.Patch(fc='#D62728', alpha=0.82, label='p<0.001 (***)'),
    mpatches.Patch(fc='#FF7F0E', alpha=0.82, label='p<0.01  (**)'),
    mpatches.Patch(fc='#2CA02C', alpha=0.82, label='p<0.05  (*)'),
    mpatches.Patch(fc='#AAAAAA', alpha=0.82, label='p≥0.05  (ns)'),
]
ax_corr.legend(handles=legend_elems2, fontsize=8, loc='lower right', framealpha=0.9)

# Panels (b), (c), ...: scatter per significant feature
panel_labels = ['(b)', '(c)', '(d)', '(e)', '(f)']
for idx, res in enumerate(reg_results):
    ax_s = fig.add_subplot(gs[0, idx + 1])
    fc   = feat_colors[idx % len(feat_colors)]
    disp = res['Display']

    ax_s.scatter(res['x_obs'], y,
                 s=50, alpha=0.70, color=fc,
                 edgecolors='white', linewidths=0.5, zorder=4)
    ax_s.plot(res['x_line'], res['y_line'],
              color='black', lw=2.0, zorder=5)
    ax_s.fill_between(res['x_line'], res['ci_low'], res['ci_high'],
                      alpha=0.25, color=fc, label='95% CI band')
    ax_s.fill_between(res['x_line'], res['pi_low'], res['pi_high'],
                      alpha=0.10, color=fc, label='95% PI band')

    pr_val, pp_val = pearsonr(res['x_obs'], y)
    sr_val, sp_val = spearmanr(res['x_obs'], y)
    textstr = (f'r = {res["R"]:+.3f} {sig_label(pp_val)}\n'
               f'R² = {res["R2"]:.3f}\n'
               f'p = {res["p_slope"]:.3e}')
    ax_s.text(0.04, 0.97, textstr,
              transform=ax_s.transAxes, fontsize=9, va='top',
              bbox=dict(boxstyle='round,pad=0.3', fc='white',
                        ec='#CCCCCC', alpha=0.90))
    ax_s.set_xlabel(disp, fontsize=10)
    ax_s.set_ylabel(TARGET_LABEL, fontsize=10)
    ax_s.set_title(f'{panel_labels[idx]} {disp}\nvs Silicide%',
                   fontsize=10, fontweight='bold')
    ax_s.legend(fontsize=8, loc='lower right', framealpha=0.85)
    ax_s.grid(True, alpha=0.25)

plt.suptitle('Mechanistic Linkage: Feature Descriptors → Silicide Phase Fraction',
             fontsize=13, fontweight='bold', y=1.02)
p4 = os.path.join(OUTPUT_DIR, 'Fig4_Combined_Publication_Panel.png')
plt.savefig(p4, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {os.path.basename(p4)}")

# ============================================================
# Export Origin data
# ============================================================
print("\n" + "=" * 80)
print("Exporting Origin-ready data ...")
print("=" * 80)

origin_path = os.path.join(ORIGIN_DIR, 'Origin_data_for_plots.xlsx')
with pd.ExcelWriter(origin_path, engine='openpyxl') as writer:

    # Sheet 1: all raw data + target
    raw_out = df[[ALLOY_COL] + FEATURE_NAMES + [TARGET]].copy()
    raw_out.to_excel(writer, sheet_name='Raw_Data', index=False)

    # Sheet 2: full correlation table
    corr_df.drop(columns=['x_obs'] if 'x_obs' in corr_df.columns else []
                 ).to_excel(writer, sheet_name='Layer1_Correlation', index=False)

    # Sheet 3+: per significant feature — scatter + fit + CI + PI
    for res in reg_results:
        feat  = res['Feature']
        short = feat[:12].replace(' ', '_').replace('/', '_')

        # scatter data
        pd.DataFrame({
            'Alloy':       alloy_ids,
            'x_observed':  res['x_obs'],
            'Silicide_pct': y,
        }).to_excel(writer, sheet_name=f'Scatter_{short}', index=False)

        # fit + CI + PI curve data
        pd.DataFrame({
            'x_fit':   res['x_line'],
            'y_fit':   res['y_line'],
            'CI_low':  res['ci_low'],
            'CI_high': res['ci_high'],
            'PI_low':  res['pi_low'],
            'PI_high': res['pi_high'],
        }).to_excel(writer, sheet_name=f'Fit_CI_PI_{short}', index=False)

        # regression statistics
        pd.DataFrame({
            'Metric':  ['Slope', 'Intercept', 'Pearson_r', 'R2',
                        'p_slope', 'Sig', 'SE_residual', 'n'],
            'Value':   [res['Slope'], res['Intercept'], res['R'], res['R2'],
                        res['p_slope'], res['Sig_slope'], res['SE_resid'], n],
        }).to_excel(writer, sheet_name=f'Stats_{short}', index=False)

print(f"  Saved: {origin_path}")

# ============================================================
# Save detailed results Excel
# ============================================================
result_path = os.path.join(OUTPUT_DIR, 'Results_Correlation_and_SLR.xlsx')
with pd.ExcelWriter(result_path, engine='openpyxl') as writer:
    corr_df.to_excel(writer, sheet_name='Layer1_All_Correlations', index=False)

    slr_summary = pd.DataFrame([{
        'Feature':    r['Feature'],
        'Display':    r['Display'],
        'Slope':      r['Slope'],
        'Intercept':  r['Intercept'],
        'Pearson_r':  r['R'],
        'R2':         r['R2'],
        'p_slope':    r['p_slope'],
        'Sig':        r['Sig_slope'],
        'SE_resid':   r['SE_resid'],
    } for r in reg_results])
    slr_summary.to_excel(writer, sheet_name='Layer2_SLR_Summary', index=False)
    df.to_excel(writer, sheet_name='Raw_Data', index=False)
print(f"  Saved: {result_path}")

# ============================================================
# Text report
# ============================================================
report_path = os.path.join(OUTPUT_DIR, 'Analysis_Report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("Feature Descriptors -> Silicide Phase Fraction | Analysis Report\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Samples  : {n}\n")
    f.write(f"Features : {len(FEATURE_NAMES)}\n")
    f.write(f"Target   : Silicide Phase Fraction  [{y.min():.2f}%, {y.max():.2f}%]\n\n")

    f.write("-" * 80 + "\n")
    f.write("LAYER 1: Pearson & Spearman Correlation\n")
    f.write("-" * 80 + "\n")
    for _, row in corr_df.iterrows():
        f.write(f"  {row['Feature']:<35}  "
                f"Pearson r={row['Pearson_r']:+.4f} "
                f"95%CI=[{row['CI95_low']:+.3f},{row['CI95_high']:+.3f}] "
                f"({row['Pearson_sig']})  "
                f"Spearman rho={row['Spearman_rho']:+.4f} ({row['Spearman_sig']})\n")

    f.write(f"\n  Significant features (p<{SIG_THRESH}): {sig_features}\n\n")

    f.write("-" * 80 + "\n")
    f.write("LAYER 2: Simple Linear Regression (significant features only)\n")
    f.write("-" * 80 + "\n")
    for res in reg_results:
        sign = '+' if res['Intercept'] >= 0 else ''
        f.write(f"\n  Feature  : {res['Feature']}\n")
        f.write(f"  Equation : Silicide% = {res['Slope']:.4f} x {res['Feature']} "
                f"{sign}{res['Intercept']:.4f}\n")
        f.write(f"  Pearson r: {res['R']:+.4f}\n")
        f.write(f"  R²       : {res['R2']:.4f}\n")
        f.write(f"  p(slope) : {res['p_slope']:.6f}  {res['Sig_slope']}\n")
        f.write(f"  SE resid : {res['SE_resid']:.4f} %\n")

print(f"  Saved: {report_path}")

# ============================================================
# Final summary
# ============================================================
print("\n" + "=" * 80)
print("Analysis Complete!")
print("=" * 80)
print(f"\n  n = {n} samples")
print(f"\n  LAYER 1 — Significant correlations (p < {SIG_THRESH}):")
for _, row in corr_df[corr_df['Significant']].iterrows():
    print(f"    {row['Feature']:<35}  "
          f"r={row['Pearson_r']:+.4f} {row['Pearson_sig']}  "
          f"95%CI=[{row['CI95_low']:+.3f}, {row['CI95_high']:+.3f}]")

print(f"\n  LAYER 2 — Simple linear regression equations:")
for res in reg_results:
    sign = '+' if res['Intercept'] >= 0 else ''
    print(f"    {res['Feature']:<35}  "
          f"Silicide% = {res['Slope']:.4f}x {sign}{res['Intercept']:.4f}  "
          f"R²={res['R2']:.4f}  p={res['p_slope']:.4e} {res['Sig_slope']}")

print(f"\n  Output: {OUTPUT_DIR}")
print(f"  Figures: 4 saved")
print(f"    Fig1_Pearson_Correlation_Overview.png")
print(f"    Fig2_Pearson_vs_Spearman.png")
print(f"    Fig3_SLR_Significant_Features_CI_PI.png")
print(f"    Fig4_Combined_Publication_Panel.png")
print("=" * 80)

Feature Descriptors -> Silicide Phase Fraction
Layer 1: Pearson & Spearman Correlation
Layer 2: Simple Linear Regression (significant features only, p<0.05)

  Loaded: 92 samples, columns: ['alloy', 'x', 'ΔSmix', 'Ω', 'Ec', 'deltaP_E13 Electron affinity', 'deltaP_热导率W/(mk)', '硅化物相体积分数']

LAYER 1: Pearson & Spearman Correlation

  Results (sorted by |Pearson r|):
  Feature                              Pearson r        p  sig              95% CI  Spearman ρ        p  sig
  ----------------------------------------------------------------------------------------------------
  Ω                                      -0.4669   0.0000  ***    [-0.613, -0.290]     -0.4942   0.0000  ***
  ΔSmix                                  -0.4366   0.0000  ***    [-0.589, -0.255]     -0.3678   0.0003  ***
  deltaP_热导率W/(mk)                       -0.3728   0.0003  ***    [-0.537, -0.182]     -0.2940   0.0044   **
  deltaP_E13 Electron affinity           -0.3678   0.0003  ***    [-0.533, -0.176]     -0.3582  